In [ ]:

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import multiprocessing as mp
import warnings
import os
import sys

warnings.filterwarnings("ignore")

OUT_DIR = os.path.join(os.path.expanduser("~"), "DEDS_output")
try:
    os.makedirs(OUT_DIR, exist_ok=True)
except PermissionError:
    OUT_DIR = os.path.join(os.getcwd(), "DEDS_output")
    os.makedirs(OUT_DIR, exist_ok=True)

print(f"Output directory: {OUT_DIR}")

MASTER_SEED = 2025
N_CORES     = max(1, mp.cpu_count() - 1)
B_DEFAULT   = 1000    # replications 
L_DEFAULT   = 99     # permutations 

# paper grids
N_GRID   = [25, 50, 100, 200]
TAU_GRID = [0.10, 0.20, 0.30, 0.40, 0.50]
NU_GRID  = [0.5, 1.0, 1.5, 2.0]
METHOD_COLS = ["DEDS", "MeanED", "PELT", "ECP"]

# =====================================================================
# SECTION 1 —  DEDS FUNCTIONS
# =====================================================================

# ── Stage-1: within-cell-line ED scan ─────────────────────────────────

def s2_psi_stage1(x: np.ndarray) -> float:
    """Pooled double-centered scale  s²_{Ψ,n}  [Eq. 6]."""
    n = len(x)
    if n < 4:
        return float(np.var(x, ddof=1)) if n > 1 else 1e-12
    diff  = np.abs(x[:, None] - x[None, :])
    hi    = diff.mean(axis=1)
    hdd   = diff.mean()
    Psi   = diff - hi[:, None] - hi[None, :] + hdd
    idx   = np.tril_indices(n, k=-1)
    return 2.0 * np.sum(Psi[idx] ** 2) / (n * (n - 1))


def energy_Ek(x: np.ndarray, k: int) -> float:
    """Two-sample energy distance  E_k  [Eq. 4]."""
    X, Y = x[:k], x[k:]
    UXY  = np.abs(X[:, None] - Y[None, :]).mean()
    UXX  = np.abs(X[:, None] - X[None, :]).mean() if k  >= 2 else 0.0
    UYY  = np.abs(Y[:, None] - Y[None, :]).mean() if len(Y) >= 2 else 0.0
    return 2.0 * UXY - UXX - UYY


def ed_scan_stage1(x: np.ndarray, eta: float = 0.10) -> dict:
    """Stage-1 ED scan statistic  Z^(ℓ)_{n,k}  [Eq. 7]."""
    n   = len(x)
    lo  = int(np.ceil(eta * n))
    hi  = int(np.floor((1.0 - eta) * n))
    K   = [k for k in range(lo, hi + 1) if k >= 2 and (n - k) >= 2]
    if not K:
        return {"K": [], "Znk": [], "khat": None, "Tn": np.nan}
    s2  = s2_psi_stage1(x)
    sp  = max(float(np.sqrt(s2)), 1e-12)
    Ka  = np.array(K, dtype=float)
    Ek  = np.array([energy_Ek(x, k) for k in K])
    Znk = Ka * (n - Ka) / (np.sqrt(2.0) * n * sp) * Ek
    best = int(np.argmax(np.abs(Znk)))
    return {"K": K, "Znk": Znk.tolist(), "khat": K[best],
            "Tn": float(np.max(np.abs(Znk)))}


# ── Stage-2: between-group DSCP statistic ─────────────────────────────

def energy_2sample(u: np.ndarray, v: np.ndarray) -> float:
    """Two-sample energy distance between vectors u and v  [Eq. 8]."""
    UUV = np.abs(u[:, None] - v[None, :]).mean()
    UUU = np.abs(u[:, None] - u[None, :]).mean() if len(u) >= 2 else 0.0
    UVV = np.abs(v[:, None] - v[None, :]).mean() if len(v) >= 2 else 0.0
    return 2.0 * UUV - UUU - UVV


def s2_psi_stage2(z: np.ndarray) -> float:
    """Pooled double-centered scale  s̃²_{Ψ,M,k}  [Eq. 9]."""
    M = len(z)
    if M < 4:
        return float(np.var(z, ddof=1)) if M > 1 else 1e-12
    diff = np.abs(z[:, None] - z[None, :])
    hi   = diff.mean(axis=1)
    hdd  = diff.mean()
    Phi  = diff - hi[:, None] - hi[None, :] + hdd
    idx  = np.tril_indices(M, k=-1)
    return 2.0 * np.sum(Phi[idx] ** 2) / (M * (M - 1))


def _get_Znk_matrix(scans: list, K: list) -> np.ndarray:
    """Build (n_cells × |K|) matrix of Stage-1 scores."""
    k_map = {k: j for j, k in enumerate(K)}
    mat   = np.full((len(scans), len(K)), np.nan)
    for i, sc in enumerate(scans):
        for j, k in enumerate(sc["K"]):
            if k in k_map:
                mat[i, k_map[k]] = sc["Znk"][j]
    return mat


def dscp_profile(scans_S: list, scans_R: list,
                  K: list = None) -> pd.DataFrame:
    """Compute the DSCP profile  Z̃_{n,k}  [Eqs. 10–11]."""
    if K is None:
        K = scans_S[0]["K"]
    mS, mR = len(scans_S), len(scans_R)
    M   = mS + mR
    ZS  = _get_Znk_matrix(scans_S, K)
    ZR  = _get_Znk_matrix(scans_R, K)
    rows = []
    for j, k in enumerate(K):
        u = ZS[:, j][~np.isnan(ZS[:, j])]
        v = ZR[:, j][~np.isnan(ZR[:, j])]
        if len(u) < 2 or len(v) < 2:
            rows.append({"k": k, "E_SR": np.nan, "Z_tilde": np.nan})
            continue
        E_SR    = energy_2sample(u, v)
        sp      = max(float(np.sqrt(s2_psi_stage2(np.concatenate([u, v])))), 1e-12)
        Z_tilde = float(np.sqrt(len(u) * len(v) / M) * E_SR / (np.sqrt(2.0) * sp))
        rows.append({"k": k, "E_SR": E_SR, "Z_tilde": Z_tilde})
    return pd.DataFrame(rows)


def dsi_profile(scans_S: list, scans_R: list,
                 K: list = None) -> pd.DataFrame:
    """Drug Sensitivity Index profile  DSI_k  [Eq. 20]."""
    if K is None:
        K = scans_S[0]["K"]
    ZS  = _get_Znk_matrix(scans_S, K)
    ZR  = _get_Znk_matrix(scans_R, K)
    dsi = np.nanmean(ZS, axis=0) - np.nanmean(ZR, axis=0)
    return pd.DataFrame({"k": K, "DSI": dsi})


def circular_block_perm(M: int, b: int,
                         rng: np.random.Generator) -> np.ndarray:
    """Circular block permutation of {0, …, M-1} with block size b."""
    starts     = list(range(0, M, b))
    perm_starts = rng.permutation(starts)
    idx = np.concatenate([(s + np.arange(b)) % M for s in perm_starts])
    return idx[:M]


def deds_test(scans_S: list, scans_R: list,
               K: list   = None,
               alpha: float = 0.05,
               L: int       = 99,
               block: int   = None,
               rng: np.random.Generator = None) -> dict:
    """Full DEDS permutation test — Algorithm 1 of the paper."""
    if rng is None:
        rng = np.random.default_rng()
    if K is None:
        K = scans_S[0]["K"]
    if not K:
        return {"reject": False, "Tn": np.nan, "c_alpha": np.nan,
                "khat": None, "DSI": np.nan}

    prof = dscp_profile(scans_S, scans_R, K)
    zt   = prof["Z_tilde"].values
    if np.all(np.isnan(zt)):
        return {"reject": False, "Tn": np.nan, "c_alpha": np.nan,
                "khat": None, "DSI": np.nan}

    Tn   = float(np.nanmax(np.abs(zt)))
    khat = K[int(np.nanargmax(np.abs(zt)))]

    # DSI at estimated locus
    ZS_k = _get_Znk_matrix(scans_S, [khat])[:, 0]
    ZR_k = _get_Znk_matrix(scans_R, [khat])[:, 0]
    DSI  = float(np.nanmean(ZS_k) - np.nanmean(ZR_k))

    # Permutation nulls — single permutation of cell-line labels
    all_scans = scans_S + scans_R
    mS        = len(scans_S)
    M_tot     = len(all_scans)
    T_perm    = np.empty(L)

    for b in range(L):
        if block is not None:
            idx = circular_block_perm(M_tot, block, rng)
        else:
            idx = rng.permutation(M_tot)
        pp        = dscp_profile([all_scans[i] for i in idx[:mS]],
                                  [all_scans[i] for i in idx[mS:]], K)
        T_perm[b] = float(np.nanmax(np.abs(pp["Z_tilde"].values)))

    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    return {"reject": bool(Tn > c_alpha), "Tn": Tn,
            "c_alpha": c_alpha, "khat": khat, "DSI": DSI}



def mean_ed_test(scans_S: list, scans_R: list,
                  K: list = None, alpha: float = 0.05) -> dict:
    """Position-wise two-sample t-test on Stage-1 scores (Mean-ED)."""
    if K is None:
        K = scans_S[0]["K"]
    ZS = _get_Znk_matrix(scans_S, K)
    ZR = _get_Znk_matrix(scans_R, K)
    pvals = []
    for j in range(len(K)):
        u = ZS[:, j][~np.isnan(ZS[:, j])]
        v = ZR[:, j][~np.isnan(ZR[:, j])]
        if len(u) < 3 or len(v) < 3:
            pvals.append(1.0); continue
        try:
            p = stats.ttest_ind(u, v, equal_var=False).pvalue
        except Exception:
            p = 1.0
        pvals.append(float(p))
    pvals  = np.array(pvals)
    min_p  = float(np.nanmin(pvals))
    khat   = K[int(np.nanargmin(pvals))]
    return {"reject": bool(min_p < alpha / max(len(K), 1)),
            "khat": khat, "Tn": float(-np.log(min_p + 1e-300))}


def pelt_diff_test(scans_S: list, scans_R: list,
                    K: list = None, alpha: float = 0.05,
                    L: int = 99,
                    rng: np.random.Generator = None) -> dict:
    """CUSUM-based differential test approximating PELT-Diff."""
    if rng is None:
        rng = np.random.default_rng()
    if K is None:
        K = scans_S[0]["K"]
    ZS   = _get_Znk_matrix(scans_S, K)
    ZR   = _get_Znk_matrix(scans_R, K)
    diff = np.nanmean(ZS, axis=0) - np.nanmean(ZR, axis=0)
    diff = np.where(np.isnan(diff), 0.0, diff)
    cumS   = np.cumsum(diff - diff.mean())
    Tn_obs = float(np.max(np.abs(cumS)))
    khat   = K[int(np.argmax(np.abs(cumS)))]

    all_sc = scans_S + scans_R
    mS     = len(scans_S)
    M_tot  = len(all_sc)
    T_perm = np.empty(L)
    for b in range(L):
        idx    = rng.permutation(M_tot)
        ZSp    = _get_Znk_matrix([all_sc[i] for i in idx[:mS]], K)
        ZRp    = _get_Znk_matrix([all_sc[i] for i in idx[mS:]], K)
        ds     = np.nanmean(ZSp, axis=0) - np.nanmean(ZRp, axis=0)
        ds     = np.where(np.isnan(ds), 0.0, ds)
        T_perm[b] = float(np.max(np.abs(np.cumsum(ds - ds.mean()))))

    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    return {"reject": bool(Tn_obs > c_alpha), "khat": khat, "Tn": Tn_obs}


def ecp_diff_test(scans_S: list, scans_R: list,
                   K: list = None, alpha: float = 0.05,
                   L: int = 99,
                   rng: np.random.Generator = None) -> dict:
    """ECP-Diff: energy-based group comparison without double-centering."""
    if rng is None:
        rng = np.random.default_rng()
    if K is None:
        K = scans_S[0]["K"]
    ZS  = _get_Znk_matrix(scans_S, K)
    ZR  = _get_Znk_matrix(scans_R, K)
    mS  = len(scans_S)
    M   = mS + len(scans_R)

    def _Evals(ZS_m, ZR_m):
        ev = []
        for j in range(len(K)):
            u = ZS_m[:, j][~np.isnan(ZS_m[:, j])]
            v = ZR_m[:, j][~np.isnan(ZR_m[:, j])]
            if len(u) < 2 or len(v) < 2:
                ev.append(0.0); continue
            pv = float(np.var(np.concatenate([u, v]), ddof=1)) + 1e-12
            ev.append(float(np.sqrt(len(u) * len(v) / M) *
                            energy_2sample(u, v) / np.sqrt(pv)))
        return np.array(ev)

    Evals  = _Evals(ZS, ZR)
    Tn_obs = float(np.max(np.abs(Evals)))
    khat   = K[int(np.argmax(np.abs(Evals)))]

    all_sc = scans_S + scans_R
    T_perm = np.empty(L)
    for b in range(L):
        idx       = rng.permutation(M)
        ZSp       = _get_Znk_matrix([all_sc[i] for i in idx[:mS]], K)
        ZRp       = _get_Znk_matrix([all_sc[i] for i in idx[mS:]], K)
        T_perm[b] = float(np.max(np.abs(_Evals(ZSp, ZRp))))

    c_alpha = float(np.nanquantile(T_perm, 1.0 - alpha))
    return {"reject": bool(Tn_obs > c_alpha), "khat": khat, "Tn": Tn_obs}


# =====================================================================
# SECTION 2 — DATA-GENERATING
# =====================================================================

def rnoise(n: int, dist: str = "normal",
           ar_rho: float = 0.0,
           rng: np.random.Generator = None) -> np.ndarray:
    """Generate n noise observations from the specified distribution."""
    if rng is None:
        rng = np.random.default_rng()
    if dist == "normal":
        raw = rng.standard_normal(n)
    elif dist == "skewnormal":
        delta = 1.0 / np.sqrt(2.0)
        raw   = (delta * np.abs(rng.standard_normal(n)) +
                 np.sqrt(1.0 - delta ** 2) * rng.standard_normal(n))
    elif dist == "exponential":
        raw = rng.exponential(scale=1.0, size=n) - 1.0
    elif dist == "t3":
        raw = rng.standard_t(df=3, size=n) / np.sqrt(3.0 / (3.0 - 2.0))
    elif dist == "laplace":
        raw = rng.laplace(loc=0.0, scale=1.0 / np.sqrt(2.0), size=n)
    elif dist == "contaminated":
        raw  = rng.standard_normal(n)
        mask = rng.random(n) < 0.05
        raw[mask] = rng.standard_normal(int(mask.sum())) * 3.0
    else:
        raise ValueError(f"Unknown distribution: {dist}")

    if ar_rho != 0.0:
        out    = np.empty(n)
        out[0] = raw[0]
        c      = float(np.sqrt(1.0 - ar_rho ** 2))
        for i in range(1, n):
            out[i] = ar_rho * out[i - 1] + c * raw[i]
        raw = out
    return raw


def add_signal(prof: np.ndarray, k_star: int,
               nu: float, signal_type: str,
               rng: np.random.Generator) -> np.ndarray:
    """Inject a distributional change at probe position k_star."""
    after = slice(k_star, len(prof))
    m     = len(prof) - k_star
    if signal_type == "mean":
        prof[after] = prof[after] + nu
    elif signal_type == "variance":
        prof[after] = prof[after] * float(np.sqrt(nu))
    elif signal_type == "shape":
        df_val      = max(float(nu) + 2.0, 3.0)
        prof[after] = (rng.standard_t(df=df_val, size=m) /
                       float(np.sqrt(df_val / (df_val - 2.0))))
    return prof


def generate_data(n: int, mS: int, mR: int,
                   k_star:      int   = None,
                   nu:          float = 1.0,
                   signal_type: str   = "mean",
                   frac_sig:    float = 1.0,
                   dist:        str   = "normal",
                   ar_rho:      float = 0.0,
                   k_star2:     int   = None,
                   nu2:         float = 1.0,
                   rng: np.random.Generator = None) -> dict:
    """Generate pharmacogenomic CNA profiles for all cell lines."""
    if rng is None:
        rng = np.random.default_rng()

    def gen_S():
        prof = rnoise(n, dist=dist, ar_rho=ar_rho, rng=rng)
        if k_star is not None and rng.random() < frac_sig:
            prof = add_signal(prof, k_star, nu, signal_type, rng)
        if k_star2 is not None:
            prof = add_signal(prof, k_star2, nu2, signal_type, rng)
        return prof

    def gen_R():
        return rnoise(n, dist=dist, ar_rho=ar_rho, rng=rng)

    return {
        "profiles_S": [gen_S() for _ in range(mS)],
        "profiles_R": [gen_R() for _ in range(mR)],
    }



def one_rep(n: int, mS: int, mR: int,
             k_star:      int   = None,
             nu:          float = 0.0,
             signal_type: str   = "mean",
             frac_sig:    float = 1.0,
             dist:        str   = "normal",
             ar_rho:      float = 0.0,
             eta:         float = 0.10,
             alpha:       float = 0.05,
             L:           int   = 99,
             block:       int   = None,
             k_star2:     int   = None,
             nu2:         float = 1.0,
             seed:        int   = None) -> dict:
    """Run one replication; return reject / khat for all four methods."""
    rng = np.random.default_rng(seed)

    dat     = generate_data(n=n, mS=mS, mR=mR,
                             k_star=k_star, nu=nu,
                             signal_type=signal_type,
                             frac_sig=frac_sig,
                             dist=dist, ar_rho=ar_rho,
                             k_star2=k_star2, nu2=nu2, rng=rng)
    scans_S = [ed_scan_stage1(p, eta=eta) for p in dat["profiles_S"]]
    scans_R = [ed_scan_stage1(p, eta=eta) for p in dat["profiles_R"]]
    K       = scans_S[0]["K"]

    null_out = {"deds": False, "mean_ed": False, "pelt": False, "ecp": False,
                "deds_err": np.nan, "mean_ed_err": np.nan,
                "pelt_err": np.nan, "ecp_err": np.nan}
    if not K:
        return null_out

    r_deds    = deds_test(scans_S, scans_R, K=K,
                           alpha=alpha, L=L, block=block, rng=rng)
    r_mean_ed = mean_ed_test(scans_S, scans_R, K=K, alpha=alpha)
    r_pelt    = pelt_diff_test(scans_S, scans_R, K=K,
                                alpha=alpha, L=L, rng=rng)
    r_ecp     = ecp_diff_test(scans_S, scans_R, K=K,
                               alpha=alpha, L=L, rng=rng)

    def loc_err(khat):
        if k_star is None or khat is None:
            return np.nan
        return abs(khat - k_star) / n

    return {
        "deds":         bool(r_deds["reject"]),
        "mean_ed":      bool(r_mean_ed["reject"]),
        "pelt":         bool(r_pelt["reject"]),
        "ecp":          bool(r_ecp["reject"]),
        "deds_err":     loc_err(r_deds["khat"]),
        "mean_ed_err":  loc_err(r_mean_ed["khat"]),
        "pelt_err":     loc_err(r_pelt["khat"]),
        "ecp_err":      loc_err(r_ecp["khat"]),
    }


def _worker(args: tuple) -> dict:
    """Top-level picklable worker for multiprocessing.Pool."""
    return one_rep(**args)


def run_study(B: int, n: int, mS: int, mR: int,
              k_star:      int   = None,
              nu:          float = 0.0,
              signal_type: str   = "mean",
              frac_sig:    float = 1.0,
              dist:        str   = "normal",
              ar_rho:      float = 0.0,
              eta:         float = 0.10,
              alpha:       float = 0.05,
              L:           int   = 99,
              block:       int   = None,
              k_star2:     int   = None,
              nu2:         float = 1.0,
              base_seed:   int   = 0,
              n_jobs:      int   = None) -> dict:
    """Run B replications in parallel; return power and localization."""
    if n_jobs is None:
        n_jobs = N_CORES

    args_list = [
        dict(n=n, mS=mS, mR=mR, k_star=k_star, nu=nu,
             signal_type=signal_type, frac_sig=frac_sig,
             dist=dist, ar_rho=ar_rho, eta=eta, alpha=alpha,
             L=L, block=block, k_star2=k_star2, nu2=nu2,
             seed=base_seed + b)
        for b in range(B)
    ]

    if n_jobs <= 1:
        reps = [_worker(a) for a in args_list]
    else:
        with mp.Pool(processes=n_jobs) as pool:
            reps = pool.map(_worker, args_list)

    def _safe_mean(key):
        vals = [r[key] for r in reps
                if isinstance(r.get(key), float) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else np.nan

    return {
        "power": {m: float(np.mean([r[m.lower()] if m == "DEDS"
                                     else r[m.lower().replace("-", "_")]
                                     for r in reps]))
                  for m in ["deds", "mean_ed", "pelt", "ecp"]},
        "power_named": {
            "DEDS":   float(np.mean([r["deds"]    for r in reps])),
            "MeanED": float(np.mean([r["mean_ed"] for r in reps])),
            "PELT":   float(np.mean([r["pelt"]    for r in reps])),
            "ECP":    float(np.mean([r["ecp"]     for r in reps])),
        },
        "loc_err": {
            "DEDS":   _safe_mean("deds_err"),
            "MeanED": _safe_mean("mean_ed_err"),
            "PELT":   _safe_mean("pelt_err"),
            "ECP":    _safe_mean("ecp_err"),
        },
    }


def _power(r: dict) -> dict:
    """Convenience accessor for the power sub-dict."""
    return r["power_named"]


# =====================================================================
# BINARY SEGMENTATION
# =====================================================================

def deds_binseg(profiles_S: list, profiles_R: list,
                 seg_s:     int   = 0,
                 seg_e:     int   = None,
                 eta:       float = 0.10,
                 alpha_seg: float = 0.05,
                 L:         int   = 49,
                 nmin:      int   = 15,
                 depth:     int   = 0,
                 max_depth: int   = 5,
                 rng: np.random.Generator = None) -> list:
    """Binary segmentation DEDS — returns list of detected probe indices."""
    if rng is None:
        rng = np.random.default_rng()
    n_prof = len(profiles_S[0])
    if seg_e is None:
        seg_e = n_prof
    if (seg_e - seg_s) < nmin or depth >= max_depth:
        return []

    sub_S   = [p[seg_s:seg_e] for p in profiles_S]
    sub_R   = [p[seg_s:seg_e] for p in profiles_R]
    scans_S = [ed_scan_stage1(p, eta=eta) for p in sub_S]
    scans_R = [ed_scan_stage1(p, eta=eta) for p in sub_R]
    K       = scans_S[0]["K"]
    if not K:
        return []

    res = deds_test(scans_S, scans_R, K=K, alpha=alpha_seg, L=L, rng=rng)
    if not res["reject"] or res["khat"] is None:
        return []

    global_k = res["khat"] + seg_s
    loci     = [global_k]
    loci    += deds_binseg(profiles_S, profiles_R,
                            seg_s=seg_s, seg_e=global_k,
                            eta=eta, alpha_seg=alpha_seg, L=L,
                            nmin=nmin, depth=depth + 1,
                            max_depth=max_depth, rng=rng)
    loci    += deds_binseg(profiles_S, profiles_R,
                            seg_s=global_k, seg_e=seg_e,
                            eta=eta, alpha_seg=alpha_seg, L=L,
                            nmin=nmin, depth=depth + 1,
                            max_depth=max_depth, rng=rng)
    return loci


# =====================================================================
# RUN ALL STUDIES
# =====================================================================

METHOD_COLORS  = {"DEDS": "#d73027", "MeanED": "#fc8d59",
                   "PELT": "#91bfdb", "ECP":    "#4575b4"}
METHOD_MARKERS = {"DEDS": "o", "MeanED": "s", "PELT": "^", "ECP": "D"}

print(f"\n{'='*60}")
print("DEDS Extended Simulation Study")
print(f"B={B_DEFAULT}, L={L_DEFAULT}, n_jobs={N_CORES}")
print(f"Sample sizes : {N_GRID}")
print(f"Change-point : {TAU_GRID}")
print(f"{'='*60}\n")

seed_cnt = MASTER_SEED

# ─────────────────────────────────────────────────────────────────────
# TYPE I ERROR (6 distributions)
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 1: Type I Error ══")

DIST_LIST = ["normal","skewnormal","exponential","t3","laplace","contaminated"]
rows1 = []
for dist in DIST_LIST:
    for n in N_GRID:
        r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                       k_star=None, nu=0.0, dist=dist,
                       L=L_DEFAULT, base_seed=seed_cnt, n_jobs=N_CORES)
        seed_cnt += B_DEFAULT
        rows1.append({"n": n, "Distribution": dist,
                       **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab1 = pd.DataFrame(rows1)
tab1.to_csv(f"{OUT_DIR}/Table_S1_TypeI_error.csv", index=False)
print(tab1.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# POWER UNDER MEAN SHIFT
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 2: Power under Mean Shift ══")

rows2 = []
for dist in ["normal","skewnormal","exponential"]:
    for n in N_GRID:
        for nu in NU_GRID:
            for tau in TAU_GRID:
                k_star = max(2, int(tau * n))
                r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                               k_star=k_star, nu=nu, signal_type="mean",
                               dist=dist, L=L_DEFAULT,
                               base_seed=seed_cnt, n_jobs=N_CORES)
                seed_cnt += B_DEFAULT
                rows2.append({"n": n, "nu": nu, "tau": tau, "dist": dist,
                               **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab2 = pd.DataFrame(rows2)
tab2.to_csv(f"{OUT_DIR}/Table_S2_Power_mean.csv", index=False)
sub2 = tab2[(tab2["n"] == 100) & (tab2["dist"] == "normal")]
print(sub2.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# STUDY 3: VARIANCE-ONLY SHIFT [NEW]
# Post-k* variance multiplied by nu; mean unchanged.
# Energy distance detects scale shifts; t-tests and PELT cannot.
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 3: Variance-Only Shift [NEW] ══")

nu_var = [1.5, 2.0, 2.5, 3.0]
rows3  = []
for dist in ["normal","skewnormal","exponential"]:
    for n in N_GRID:
        for nu in nu_var:
            for tau in TAU_GRID:
                k_star = max(2, int(tau * n))
                r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                               k_star=k_star, nu=nu, signal_type="variance",
                               dist=dist, L=L_DEFAULT,
                               base_seed=seed_cnt, n_jobs=N_CORES)
                seed_cnt += B_DEFAULT
                rows3.append({"n": n, "nu_var": nu, "tau": tau, "dist": dist,
                               **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab3 = pd.DataFrame(rows3)
tab3.to_csv(f"{OUT_DIR}/Table_S3_Power_variance.csv", index=False)
sub3 = tab3[(tab3["n"] == 100) & (tab3["dist"] == "normal")]
print(sub3.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# SHAPE-ONLY CHANGE 
# Pre-locus: N(0,1). Post-locus: t(df) with unit variance.
# Same mean AND variance — only tail behavior differs.
# t-tests and PELT are theoretically blind here.
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 4: Shape-Only Change [NEW] ══")

df_seq = [3, 4, 6, 10]  # df = nu+2  where nu = 1,2,4,8
rows4  = []
for n in N_GRID:
    for df_val in df_seq:
        nu_shape = float(df_val - 2)
        for tau in [0.30, 0.50]:
            k_star = max(2, int(tau * n))
            r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                           k_star=k_star, nu=nu_shape, signal_type="shape",
                           dist="normal", L=L_DEFAULT,
                           base_seed=seed_cnt, n_jobs=N_CORES)
            seed_cnt += B_DEFAULT
            rows4.append({"n": n, "df": df_val, "tau": tau,
                           **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab4 = pd.DataFrame(rows4)
tab4.to_csv(f"{OUT_DIR}/Table_S4_Power_shape.csv", index=False)
print(tab4.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# PARTIAL CONTAMINATION 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 5: Partial Contamination [NEW] ══")

frac_grid = [0.30, 0.50, 0.70, 1.00]
rows5     = []
for n in [50, 100, 200]:
    for nu in [1.0, 1.5, 2.0]:
        for fs in frac_grid:
            k_star = max(2, int(0.50 * n))
            r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                           k_star=k_star, nu=nu, signal_type="mean",
                           frac_sig=fs, dist="normal", L=L_DEFAULT,
                           base_seed=seed_cnt, n_jobs=N_CORES)
            seed_cnt += B_DEFAULT
            rows5.append({"n": n, "nu": nu, "frac_sig": fs,
                           **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab5 = pd.DataFrame(rows5)
tab5.to_csv(f"{OUT_DIR}/Table_S5_Partial_cont.csv", index=False)
print(tab5[tab5["n"] == 100].to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# ASYMMETRIC GROUP SIZES 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 6: Asymmetric Group Sizes [NEW] ══")

M_total  = 30
rho_grid = [0.20, 0.30, 0.40, 0.50]
rows6    = []
for rho in rho_grid:
    mS_i = max(3, int(round(rho * M_total)))
    mR_i = M_total - mS_i
    for nu in [0.0] + NU_GRID:
        k_star = None if nu == 0.0 else int(0.5 * 100)
        r = run_study(B=B_DEFAULT, n=100, mS=mS_i, mR=mR_i,
                       k_star=k_star, nu=nu, signal_type="mean",
                       dist="normal", L=L_DEFAULT,
                       base_seed=seed_cnt, n_jobs=N_CORES)
        seed_cnt += B_DEFAULT
        rows6.append({"rho": rho, "mS": mS_i, "mR": mR_i, "nu": nu,
                       **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab6 = pd.DataFrame(rows6)
tab6.to_csv(f"{OUT_DIR}/Table_S6_Asymm_groups.csv", index=False)
print(tab6.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# AR(1) WITHIN-PROFILE DEPENDENCE 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 7: AR(1) Dependence & Block Permutation [NEW] ══")

ar_grid  = [0.0, 0.30, 0.50, 0.70]
nu_grid7 = [0.0, 1.0, 1.5]
rows7    = []
for ar in ar_grid:
    for nu in nu_grid7:
        k_star   = None if nu == 0.0 else int(0.5 * 100)
        block_sz = int(np.ceil(np.sqrt(100)))
        r_s = run_study(B=200, n=100, mS=15, mR=15,
                         k_star=k_star, nu=nu, signal_type="mean",
                         dist="normal", ar_rho=ar,
                         block=None, L=L_DEFAULT,
                         base_seed=seed_cnt, n_jobs=N_CORES)
        seed_cnt += 200
        r_b = run_study(B=200, n=100, mS=15, mR=15,
                         k_star=k_star, nu=nu, signal_type="mean",
                         dist="normal", ar_rho=ar,
                         block=block_sz, L=L_DEFAULT,
                         base_seed=seed_cnt, n_jobs=N_CORES)
        seed_cnt += 200
        rows7.append({
            "ar_rho": ar, "nu": nu,
            "DEDS_simple":   round(_power(r_s)["DEDS"],   3),
            "DEDS_block":    round(_power(r_b)["DEDS"],   3),
            "MeanED_simple": round(_power(r_s)["MeanED"], 3),
            "ECP_simple":    round(_power(r_s)["ECP"],    3),
        })

tab7 = pd.DataFrame(rows7)
tab7.to_csv(f"{OUT_DIR}/Table_S7_AR1_block.csv", index=False)
print(tab7.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# MULTIPLE SIMULTANEOUS LOCI 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 8: Multiple Simultaneous Loci [NEW] ══")

B8         = 200
alpha_seg8 = 0.05 / 3.0
rows8      = []
for n in [100, 200]:
    for nu in [1.0, 1.5]:
        k1  = max(2, int(0.25 * n))
        k2  = max(2, int(0.65 * n))
        tol = max(2, int(0.05 * n))
        both_det, k1_only, k2_only, neither, n_det_all = [], [], [], [], []
        for b in range(B8):
            rng = np.random.default_rng(seed_cnt + b)
            dat = generate_data(n=n, mS=15, mR=15,
                                 k_star=k1, nu=nu, signal_type="mean",
                                 k_star2=k2, nu2=nu, dist="normal", rng=rng)
            loci   = deds_binseg(dat["profiles_S"], dat["profiles_R"],
                                  alpha_seg=alpha_seg8, L=49, nmin=10, rng=rng)
            k1_hit = any(abs(p - k1) <= tol for p in loci)
            k2_hit = any(abs(p - k2) <= tol for p in loci)
            both_det.append(k1_hit and k2_hit)
            k1_only.append(k1_hit and not k2_hit)
            k2_only.append(not k1_hit and k2_hit)
            neither.append(not k1_hit and not k2_hit)
            n_det_all.append(len(loci))
        seed_cnt += B8
        rows8.append({
            "n": n, "nu": nu,
            "P_both":    round(float(np.mean(both_det)),  3),
            "P_k1_only": round(float(np.mean(k1_only)),   3),
            "P_k2_only": round(float(np.mean(k2_only)),   3),
            "P_neither": round(float(np.mean(neither)),    3),
            "mean_n_det": round(float(np.mean(n_det_all)), 2),
        })

tab8 = pd.DataFrame(rows8)
tab8.to_csv(f"{OUT_DIR}/Table_S8_Multi_loci.csv", index=False)
print(tab8.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# POWER vs NUMBER OF CELL LINES M 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 9: Power vs. Number of Cell Lines M [NEW] ══")

M_grid   = [10, 16, 20, 30, 40, 60, 80]
nu_grid9 = [0.5, 1.0, 1.5]
rows9    = []
for M_j in M_grid:
    mS_j = M_j // 2
    mR_j = M_j - mS_j
    for nu in nu_grid9:
        r = run_study(B=B_DEFAULT, n=100, mS=mS_j, mR=mR_j,
                       k_star=50, nu=nu, signal_type="mean",
                       dist="normal", L=L_DEFAULT,
                       base_seed=seed_cnt, n_jobs=N_CORES)
        seed_cnt += B_DEFAULT
        rows9.append({"M": M_j, "nu": nu,
                       **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab9 = pd.DataFrame(rows9)
tab9.to_csv(f"{OUT_DIR}/Table_S9_Power_vs_M.csv", index=False)
print(tab9.to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# SMALL SAMPLE & TRIMMING PARAMETER 
# ─────────────────────────────────────────────────────────────────────
print("══ STUDY 10: Small Sample & Trimming [NEW] ══")

eta_grid = [0.05, 0.10, 0.15, 0.20]
mS_sm    = [5, 8, 12]
rows10   = []
for n in [25, 50]:
    for mS_i in mS_sm:
        for eta in eta_grid:
            for nu in [0.0, 1.0, 1.5]:
                k_star = None if nu == 0.0 else max(2, int(0.5 * n))
                r = run_study(B=200, n=n, mS=mS_i, mR=mS_i,
                               k_star=k_star, nu=nu, signal_type="mean",
                               dist="normal", eta=eta, L=L_DEFAULT,
                               base_seed=seed_cnt, n_jobs=N_CORES)
                seed_cnt += 200
                rows10.append({"n": n, "mS": mS_i, "eta": eta, "nu": nu,
                                **{m: round(_power(r)[m], 3) for m in METHOD_COLS}})

tab10 = pd.DataFrame(rows10)
tab10.to_csv(f"{OUT_DIR}/Table_S10_Small_sample.csv", index=False)
print(tab10[(tab10["n"] == 50) & (tab10["nu"] == 1.5)].to_string(index=False), "\n")

# ─────────────────────────────────────────────────────────────────────
# EXTENDED LOCALIZATION ACCURACY (mean + variance signals)
# ─────────────────────────────────────────────────────────────────────
print("══ Extended Localization Accuracy ══")

rows_loc = []
for dist in ["normal","skewnormal","exponential"]:
    for sig_type in ["mean","variance"]:
        for n in N_GRID:
            for nu in NU_GRID:
                for tau in TAU_GRID:
                    k_star = max(2, int(tau * n))
                    r = run_study(B=B_DEFAULT, n=n, mS=15, mR=15,
                                   k_star=k_star, nu=nu,
                                   signal_type=sig_type,
                                   dist=dist, L=L_DEFAULT,
                                   base_seed=seed_cnt, n_jobs=N_CORES)
                    seed_cnt += B_DEFAULT
                    rows_loc.append({
                        "Signal": sig_type, "dist": dist,
                        "n": n, "nu": nu, "tau": tau,
                        **{f"{m}_err": round(r["loc_err"][m], 4)
                           for m in METHOD_COLS}
                    })

tab_loc = pd.DataFrame(rows_loc)
tab_loc.to_csv(f"{OUT_DIR}/Table_S11_Localization.csv", index=False)
sub_loc = tab_loc[(tab_loc["n"] == 100) & (tab_loc["dist"] == "normal")]
print(sub_loc.to_string(index=False), "\n")

print("Generating plots …")

pdf_path = f"{OUT_DIR}/DEDS_extended_simulation.pdf"

def _facet_power(data: pd.DataFrame,
                  x_col: str, x_label: str,
                  row_col: str, col_col: str,
                  title: str, subtitle: str = "",
                  hline: float = 0.05) -> plt.Figure:
    """Generic faceted power-curve figure."""
    row_vals = sorted(data[row_col].unique())
    col_vals = sorted(data[col_col].unique())
    nr, nc   = len(row_vals), len(col_vals)
    fig, axes = plt.subplots(nr, nc,
                              figsize=(3.2 * nc, 2.8 * nr),
                              sharex=True, sharey=True,
                              constrained_layout=True)
    axes = np.array(axes).reshape(nr, nc)
    for i, rv in enumerate(row_vals):
        for j, cv in enumerate(col_vals):
            ax  = axes[i, j]
            sub = data[(data[row_col] == rv) & (data[col_col] == cv)]
            for m in METHOD_COLS:
                if m not in sub.columns:
                    continue
                ax.plot(sub[x_col], sub[m],
                        color=METHOD_COLORS[m],
                        marker=METHOD_MARKERS[m],
                        markersize=4, linewidth=1.4, label=m)
            ax.axhline(hline, color="grey", linestyle="--", linewidth=0.8)
            ax.set_ylim(-0.02, 1.05)
            if i == 0:
                ax.set_title(f"{col_col}={cv}", fontsize=8, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{row_col}={rv}\nPower", fontsize=7)
            ax.set_xlabel(x_label, fontsize=7)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", ncol=4, fontsize=8, frameon=True)
    fig.suptitle(f"{title}\n{subtitle}", fontsize=10, fontweight="bold", y=1.01)
    return fig


with PdfPages(pdf_path) as pdf:

    # ── Plot 1: Type I error heatmap ───────────────────────────────────
    dist_labels = {"normal": "N(0,1)", "skewnormal": "SN(0,1,1)",
                   "exponential": "Exp(1)", "t3": "t(3)",
                   "laplace": "Laplace", "contaminated": "Contam."}
    plot1 = tab1.copy()
    plot1["dist_lab"] = plot1["Distribution"].map(dist_labels)
    pivot  = plot1.pivot_table(index="dist_lab", columns="n",
                                values="DEDS", aggfunc="mean")
    fig1, ax1 = plt.subplots(figsize=(10, 5), constrained_layout=True)
    im = ax1.imshow(pivot.values, cmap="RdYlGn_r",
                    vmin=0.02, vmax=0.10, aspect="auto")
    ax1.set_xticks(range(len(pivot.columns)))
    ax1.set_xticklabels([f"n={c}" for c in pivot.columns])
    ax1.set_yticks(range(len(pivot.index)))
    ax1.set_yticklabels(pivot.index)
    for ii in range(pivot.shape[0]):
        for jj in range(pivot.shape[1]):
            ax1.text(jj, ii, f"{pivot.values[ii, jj]:.3f}",
                     ha="center", va="center", fontsize=9)
    plt.colorbar(im, ax=ax1, label="DEDS Rejection Rate")
    ax1.set_title("Study 1: Empirical Type I Error — DEDS  (nominal α = 0.05)\n"
                  "M = 30 (mS = mR = 15), 1000 replications",
                  fontweight="bold")
    pdf.savefig(fig1, bbox_inches="tight"); plt.close(fig1)

    # ── Plot 2: Power under mean shift (normal) ────────────────────────
    sub2 = tab2[tab2["dist"] == "normal"].copy()
    sub2["n_lab"]  = "n=" + sub2["n"].astype(str)
    sub2["nu_lab"] = "ν=" + sub2["nu"].astype(str)
    fig2 = _facet_power(sub2, x_col="tau",
                         x_label="Change-Point Location τ*",
                         row_col="n_lab", col_col="nu_lab",
                         title="Study 2: Power under Mean Shift  [N(0,1)]",
                         subtitle="M=30, 1000 replications")
    pdf.savefig(fig2, bbox_inches="tight"); plt.close(fig2)

    # ── Plot 3: Variance-only shift ─────────────────────────────────────
    sub3 = tab3[tab3["dist"] == "normal"].copy()
    sub3["n_lab"]   = "n=" + sub3["n"].astype(str)
    sub3["nuv_lab"] = "σ²×" + sub3["nu_var"].astype(str)
    fig3 = _facet_power(sub3, x_col="tau",
                         x_label="Change-Point Location τ*",
                         row_col="n_lab", col_col="nuv_lab",
                         title="Study 3: Power under Variance-Only Shift  [NEW]",
                         subtitle="No mean change; t-test & PELT blind")
    pdf.savefig(fig3, bbox_inches="tight"); plt.close(fig3)

    # ── Plot 4: Shape-only change ────────────────────────────────────────
    sub4 = tab4.copy()
    sub4["n_lab"]   = "n=" + sub4["n"].astype(str)
    sub4["tau_lab"] = "τ*=" + sub4["tau"].astype(str)
    fig4 = _facet_power(sub4, x_col="df",
                         x_label="Degrees of Freedom (smaller = heavier tail)",
                         row_col="n_lab", col_col="tau_lab",
                         title="Study 4: Power under Shape-Only Change  [NEW]",
                         subtitle="Same mean & variance; N(0,1)→t(df)/σ")
    for row in fig4.axes:
        row.invert_xaxis()
    pdf.savefig(fig4, bbox_inches="tight"); plt.close(fig4)

    # ── Plot 5: Partial contamination ──────────────────────────────────
    sub5 = tab5[tab5["n"] == 100].copy()
    sub5["nu_lab"] = "ν=" + sub5["nu"].astype(str)
    nu_vals5 = sorted(sub5["nu_lab"].unique())
    fig5, ax5s = plt.subplots(1, len(nu_vals5), figsize=(4 * len(nu_vals5), 4),
                               sharey=True, constrained_layout=True)
    for jj, nl in enumerate(nu_vals5):
        ax  = ax5s[jj]
        sub = sub5[sub5["nu_lab"] == nl]
        for m in METHOD_COLS:
            ax.plot(sub["frac_sig"], sub[m],
                    color=METHOD_COLORS[m], marker=METHOD_MARKERS[m],
                    markersize=5, linewidth=1.5, label=m)
        ax.axhline(0.05, color="grey", linestyle="--", linewidth=0.8)
        ax.set_xlim(0.25, 1.05); ax.set_ylim(-0.02, 1.05)
        ax.set_title(nl, fontweight="bold")
        ax.set_xlabel("Fraction of S with signal")
        ax.set_xticks([0.3, 0.5, 0.7, 1.0])
    ax5s[0].set_ylabel("Power")
    h, l = ax5s[0].get_legend_handles_labels()
    fig5.legend(h, l, loc="lower center", ncol=4, fontsize=9)
    fig5.suptitle("Study 5: Partial Contamination [NEW]\n"
                  "n=100, M=30. Heterogeneous sensitivity class.", fontweight="bold")
    pdf.savefig(fig5, bbox_inches="tight"); plt.close(fig5)

    # ── Plot 6: Asymmetric groups ───────────────────────────────────────
    sub6 = tab6[tab6["nu"] > 0].copy()
    sub6["rho_lab"] = ("ρ=" + sub6["rho"].astype(str) +
                        "  (mS=" + sub6["mS"].astype(str) + ")")
    rho_vals = sorted(sub6["rho_lab"].unique())
    fig6, ax6s = plt.subplots(1, len(rho_vals), figsize=(4 * len(rho_vals), 4),
                               sharey=True, constrained_layout=True)
    for jj, rl in enumerate(rho_vals):
        ax  = ax6s[jj]
        sub = sub6[sub6["rho_lab"] == rl]
        for m in METHOD_COLS:
            ax.plot(sub["nu"], sub[m],
                    color=METHOD_COLORS[m], marker=METHOD_MARKERS[m],
                    markersize=5, linewidth=1.5, label=m)
        ax.axhline(0.05, color="grey", linestyle="--", linewidth=0.8)
        ax.set_ylim(-0.02, 1.05)
        ax.set_title(rl, fontsize=8, fontweight="bold")
        ax.set_xlabel("CNA Contrast ν")
    ax6s[0].set_ylabel("Power")
    h, l = ax6s[0].get_legend_handles_labels()
    fig6.legend(h, l, loc="lower center", ncol=4, fontsize=9)
    fig6.suptitle("Study 6: Asymmetric Group Sizes [NEW]\n"
                  "n=100, M=30. Varying mS/M from 0.20 to 0.50.",
                  fontweight="bold")
    pdf.savefig(fig6, bbox_inches="tight"); plt.close(fig6)

    # ── Plot 7: AR(1) dependence ────────────────────────────────────────
    types7 = [(0.0, "Size (H0)"), (1.0, "Power (ν=1.0)"), (1.5, "Power (ν=1.5)")]
    fig7, ax7s = plt.subplots(1, 3, figsize=(12, 4),
                               sharey=True, constrained_layout=True)
    for jj, (nu_v, lbl) in enumerate(types7):
        ax  = ax7s[jj]
        sub = tab7[tab7["nu"] == nu_v]
        ax.plot(sub["ar_rho"], sub["DEDS_simple"],   "o-",
                color="#d73027", linewidth=1.5, label="DEDS (simple)")
        ax.plot(sub["ar_rho"], sub["DEDS_block"],    "s--",
                color="#313695", linewidth=1.5, label="DEDS (block)")
        ax.plot(sub["ar_rho"], sub["MeanED_simple"], "^:",
                color="#fc8d59", linewidth=1.5, label="Mean-ED")
        ax.plot(sub["ar_rho"], sub["ECP_simple"],    "D:",
                color="#4575b4", linewidth=1.5, label="ECP")
        ax.axhline(0.05, color="grey", linestyle="--", linewidth=0.8)
        ax.set_xlim(-0.05, 0.75); ax.set_ylim(-0.02, 1.05)
        ax.set_xlabel("AR(1) coefficient ρ")
        ax.set_title(lbl, fontweight="bold")
    ax7s[0].set_ylabel("Rejection Rate")
    h, l = ax7s[0].get_legend_handles_labels()
    fig7.legend(h, l, loc="lower center", ncol=4, fontsize=9)
    fig7.suptitle("Study 7: AR(1) Within-Profile Dependence [NEW]\n"
                  "n=100, M=30. Block perm corrects inflation.",
                  fontweight="bold")
    pdf.savefig(fig7, bbox_inches="tight"); plt.close(fig7)

    # ── Plot 8: Multiple loci stacked bar ──────────────────────────────
    fig8, ax8 = plt.subplots(figsize=(8, 5), constrained_layout=True)
    labels8 = [f"n={r.n}, ν={r.nu}" for r in tab8.itertuples()]
    bot     = np.zeros(len(tab8))
    cols8   = {"P_both": "#1a9850", "P_k1_only": "#a6d96a",
               "P_k2_only": "#fdae61", "P_neither": "#d73027"}
    lbls8   = {"P_both": "Both", "P_k1_only": "Locus 1 only",
               "P_k2_only": "Locus 2 only", "P_neither": "Neither"}
    for col, color in cols8.items():
        vals = tab8[col].values
        ax8.bar(labels8, vals, bottom=bot, color=color, label=lbls8[col], width=0.6)
        bot += vals
    ax8.set_ylabel("Probability"); ax8.set_ylim(0, 1.05)
    ax8.legend(loc="upper right", fontsize=9)
    plt.setp(ax8.get_xticklabels(), rotation=20, ha="right", fontsize=9)
    ax8.set_title("Study 8: Binary Segmentation — Two Simultaneous Loci [NEW]\n"
                  "τ1*=0.25, τ2*=0.65;  αseg=0.05/3;  200 replications",
                  fontweight="bold")
    pdf.savefig(fig8, bbox_inches="tight"); plt.close(fig8)

    # ── Plot 9: Power vs M ──────────────────────────────────────────────
    fig9, ax9s = plt.subplots(1, 3, figsize=(12, 4),
                               sharey=True, constrained_layout=True)
    for jj, nu_v in enumerate(nu_grid9):
        ax  = ax9s[jj]
        sub = tab9[tab9["nu"] == nu_v]
        for m in METHOD_COLS:
            ax.plot(sub["M"], sub[m],
                    color=METHOD_COLORS[m], marker=METHOD_MARKERS[m],
                    markersize=5, linewidth=1.5, label=m)
        ax.axhline(0.05, color="grey", linestyle="--", linewidth=0.8)
        ax.set_ylim(-0.02, 1.05)
        ax.set_xlabel("Number of Cell Lines M")
        ax.set_title(f"ν={nu_v}", fontweight="bold")
    ax9s[0].set_ylabel("Power")
    h, l = ax9s[0].get_legend_handles_labels()
    fig9.legend(h, l, loc="lower center", ncol=4, fontsize=9)
    fig9.suptitle("Study 9: Power vs. Number of Cell Lines M [NEW]\n"
                  "n=100, τ*=0.5. Theorem 3.7: power → 1 as M → ∞.",
                  fontweight="bold")
    pdf.savefig(fig9, bbox_inches="tight"); plt.close(fig9)

    # ── Plot 10: Small sample & trimming ───────────────────────────────
    for n_v in [25, 50]:
        sub10 = tab10[(tab10["n"] == n_v) & (tab10["nu"] > 0)].copy()
        if sub10.empty:
            continue
        sub10["mS_lab"] = "mS=mR=" + sub10["mS"].astype(str)
        sub10["nu_lab"] = "ν=" + sub10["nu"].astype(str)
        row_vals10 = sorted(sub10["mS_lab"].unique())
        col_vals10 = sorted(sub10["nu_lab"].unique())
        fig10, ax10 = plt.subplots(len(row_vals10), len(col_vals10),
                                    figsize=(4 * len(col_vals10),
                                             3 * len(row_vals10)),
                                    sharex=True, sharey=True,
                                    constrained_layout=True)
        ax10 = np.array(ax10).reshape(len(row_vals10), len(col_vals10))
        for ii, rl in enumerate(row_vals10):
            for jj, nl in enumerate(col_vals10):
                ax   = ax10[ii, jj]
                sub_ = sub10[(sub10["mS_lab"] == rl) & (sub10["nu_lab"] == nl)]
                for m in METHOD_COLS:
                    ax.plot(sub_["eta"], sub_[m],
                            color=METHOD_COLORS[m], marker=METHOD_MARKERS[m],
                            markersize=4, linewidth=1.4, label=m)
                ax.axhline(0.05, color="grey", linestyle="--", linewidth=0.8)
                ax.set_ylim(-0.02, 1.05)
                if ii == 0:
                    ax.set_title(nl, fontsize=9, fontweight="bold")
                if jj == 0:
                    ax.set_ylabel(f"{rl}\nPower", fontsize=8)
                ax.set_xlabel("Trimming η", fontsize=8)
        h, l = ax10[0, 0].get_legend_handles_labels()
        fig10.legend(h, l, loc="lower center", ncol=4, fontsize=9)
        fig10.suptitle(f"Study 10: Small Sample & Trimming [NEW]  (n={n_v})\n"
                       "200 replications.", fontweight="bold")
        pdf.savefig(fig10, bbox_inches="tight"); plt.close(fig10)

    # ── Plot 11: Localization error (mean vs variance) ─────────────────
    sub_l = tab_loc[(tab_loc["n"] == 100) & (tab_loc["tau"] == 0.50)].copy()
    dist_labels2 = {"normal": "N(0,1)", "skewnormal": "SN(0,1,1)",
                    "exponential": "Exp(1)"}
    sub_l["dist_lab"]   = sub_l["dist"].map(dist_labels2)
    sub_l["signal_lab"] = sub_l["Signal"].map({"mean": "Mean Shift",
                                                "variance": "Variance Shift"})
    fig11, ax11 = plt.subplots(2, 3, figsize=(12, 7),
                                sharex=True, sharey=True,
                                constrained_layout=True)
    sig_rows  = ["Mean Shift", "Variance Shift"]
    dist_cols = ["N(0,1)", "SN(0,1,1)", "Exp(1)"]
    for ii, rv in enumerate(sig_rows):
        for jj, cv in enumerate(dist_cols):
            ax   = ax11[ii, jj]
            sub_ = sub_l[(sub_l["signal_lab"] == rv) &
                          (sub_l["dist_lab"]  == cv)]
            for m in METHOD_COLS:
                ec = f"{m}_err"
                if ec not in sub_.columns:
                    continue
                ax.plot(sub_["nu"], sub_[ec],
                        color=METHOD_COLORS[m], marker=METHOD_MARKERS[m],
                        markersize=5, linewidth=1.5, label=m)
            ax.set_ylim(-0.005, 0.22)
            if ii == 0:
                ax.set_title(cv, fontsize=9, fontweight="bold")
            if jj == 0:
                ax.set_ylabel(f"{rv}\n|k̂−k*|/n", fontsize=8)
            ax.set_xlabel("ν", fontsize=8)
    h, l = ax11[0, 0].get_legend_handles_labels()
    fig11.legend(h, l, loc="lower center", ncol=4, fontsize=9)
    fig11.suptitle("Extended Localization Accuracy: Mean vs Variance Shift\n"
                   "n=100, τ*=0.50, M=30. Smaller is better.",
                   fontweight="bold")
    pdf.savefig(fig11, bbox_inches="tight"); plt.close(fig11)

print(f"All plots  → {pdf_path}")

# ── save all tables ────────────────────────────────────────────────────
print(f"All tables → {OUT_DIR}/")

# ── session info ───────────────────────────────────────────────────────
info = {
    "Python":  sys.version.split()[0],
    "NumPy":   np.__version__,
    "Pandas":  pd.__version__,
    "SciPy":   stats.__version__,
    "n_cores": N_CORES,
    "B":       B_DEFAULT,
    "L":       L_DEFAULT,
    "OUT_DIR": OUT_DIR,
}
pd.DataFrame([info]).to_csv(f"{OUT_DIR}/session_info.csv", index=False)
print("\nSession info:")
for k, v in info.items():
    print(f"  {k}: {v}")
print("\nDone.")

Output directory: /home/jovyan/DEDS_output

DEDS Extended Simulation Study
B=1000, L=99, n_jobs=191
Sample sizes : [25, 50, 100, 200]
Change-point : [0.1, 0.2, 0.3, 0.4, 0.5]

══ STUDY 1: Type I Error ══
  n Distribution  DEDS  MeanED  PELT   ECP
 25       normal 0.076   0.017 0.058 0.059
 50       normal 0.070   0.012 0.063 0.065
100       normal 0.063   0.004 0.063 0.063
200       normal 0.053   0.002 0.056 0.055
 25   skewnormal 0.058   0.011 0.053 0.058
 50   skewnormal 0.061   0.010 0.055 0.049
100   skewnormal 0.064   0.008 0.073 0.065
200   skewnormal 0.061   0.006 0.076 0.061
 25  exponential 0.063   0.005 0.059 0.069
 50  exponential 0.055   0.005 0.055 0.049
100  exponential 0.045   0.001 0.056 0.050
200  exponential 0.059   0.007 0.055 0.060
 25           t3 0.048   0.009 0.068 0.056
 50           t3 0.067   0.011 0.077 0.066
100           t3 0.046   0.009 0.055 0.054
200           t3 0.069   0.004 0.049 0.068
 25      laplace 0.071   0.018 0.062 0.066
 50      laplace 0.069